# Demo: 2018 Aude Flood

## 1. Setup

In [ ]:
import sys
sys.path.append("../src")

In [ ]:
import twitter_dataset as td
import rioda_python_driver as riopy
import crisis_image_benchmarks_classifier as cibClassifier
import crisis_image_benchmarks_middleware as cibMiddleware

In [ ]:
import matplotlib.pyplot as plt
from dateutil.parser import parse
from tqdm import tqdm

## 2. Load 2018 Aude Flood Twitter Dataset

In [ ]:
# load twitter dataset from .json
tweets = td.load_tweets_from_json(dataset = '2018 Aude Flood')
print('Count of Tweets :', len(tweets))

In [ ]:
# show the first tweet
print('Created at :', parse(tweets[24]['created_at']))
print('Place concerned :', tweets[24]['place_concerned'])
print('Full content :', tweets[24]['full_text'])
imgs = td.fetch_imgs(tweets[24], False, dataset = '2018 Aude Flood')
for img in imgs:
    plt.figure()
    plt.imshow(img)

## 3. Image interpretation

In [ ]:
# Load pretrained models
pretrain = cibClassifier.load_models()

In [ ]:
# use pretrained models to classify image
results = td.cib_classify_tweet(
    tweets[24],
    pretrain = pretrain,
    online = False,
    dataset = '2018 Aude Flood'
)
results

## 4. Move from labels to concepts

In [ ]:
# move from labels to concepts
concepts = td.cib_translate_tweet(tweets[24], results)
concepts

In [ ]:
# move from labels to concepts
print('Count of concepts to be instantiated :', len(concepts))

## 5. Instantiate concepts in R-IODA

In [ ]:
# initialization
uri = 'neo4j://localhost:7687'
userName = 'neo4j'
password = 'neo4j'
driver = riopy.initialize(uri, userName, password)

In [ ]:
# get the name of current knowledge space and collaboration
with driver.session() as session:
    currentKnowledgeSpaceName = session.read_transaction(riopy.get_current_knowledge_space_name).lower()
    currentCollaborationName = session.read_transaction(riopy.get_current_collaboration_name).lower()
print('The Name of Current Knowledge Space is :', currentKnowledgeSpaceName)
print('The Name of Current Collaboration is :', currentCollaborationName)
driver.close()

In [ ]:
# instantiate concepts
with driver.session() as session:
    for concept in tqdm(concepts):
        riopy.instantiate_concept(session, concept, currentKnowledgeSpaceName, currentCollaborationName, verbose = False)
driver.close()

## 6. End to end

In [ ]:
# load twitter dataset from .json
tweets = td.load_tweets_from_json(dataset = '2018 Aude Flood')

# Load pretrained models
pretrain = cibClassifier.load_models()

# initialization
uri = 'neo4j://localhost:7687'
userName = 'neo4j'
password = 'neo4j'
driver = riopy.initialize(uri, userName, password)

# interpret tweets
with driver.session() as session:
    currentKnowledgeSpaceName = session.read_transaction(riopy.get_current_knowledge_space_name).lower()
    currentCollaborationName = session.read_transaction(riopy.get_current_collaboration_name).lower()
    for tweet in tqdm(tweets):
        td.cib_interpret_tweet(
            session,
            currentKnowledgeSpaceName,
            currentCollaborationName,
            tweet,
            pretrain = pretrain,
            online = False,
            dataset = '2018 Aude Flood',
            verbose = False
        )
driver.close()